# Notebook 05v2 — Prospective State-Level Disaster Risk Forecasting

**Business case:** Before any disaster is declared, FEMA regional offices and state emergency managers
need to know which states carry the highest risk of a major PA-eligible disaster in the coming year.
This notebook builds a **genuinely prospective** model: all features are available *before* the
forecast year begins — no incident type, no declaration lag, no project counts.

**Target:** `max_tier` — the highest disaster funding tier experienced by a given state in a given year.
- 0 = No significant disaster (or only Minor <$1M events)
- 1 = Moderate ($1M–$50M)
- 2 = Major ($50M–$500M)
- 3 = Catastrophic (>$500M)

**Unit of analysis:** one row = one state × one year (~1,200 rows total)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import sys
sys.path.append('../')
from utils import classification_metrics

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, classification_report
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.base import clone

sns.set_theme(style='whitegrid')
PROCESSED = '../data/processed/'

disas = pd.read_csv(PROCESSED + 'cleaned_disaster_level.csv', low_memory=False)
print('Disaster-level shape:', disas.shape)
print('Years:', disas['incident_year'].min(), '–', disas['incident_year'].max())

## 5v2.1 Build State × Year Panel

Create a complete grid of every state × every year in the dataset.
State-years with no PA-eligible disasters get `max_tier = 0`.
This represents the **no significant disaster** outcome the model must also predict.

In [ ]:
states = sorted(disas['stateAbbreviation'].dropna().unique())
years  = list(range(int(disas['incident_year'].min()),
                    int(disas['incident_year'].max()) + 1))

# Complete grid
panel = pd.DataFrame(
    [(s, y) for s in states for y in years],
    columns=['stateAbbreviation', 'incident_year']
)

# Aggregate disasters per state-year
disaster_agg = (
    disas.groupby(['stateAbbreviation', 'incident_year'])
    .agg(
        max_tier    = ('funding_tier', 'max'),
        n_disasters = ('disasterNumber', 'count'),
    )
    .reset_index()
)

# Merge — no-disaster years get NaN → fill with 0
panel = panel.merge(disaster_agg, on=['stateAbbreviation', 'incident_year'], how='left')
panel['max_tier']    = panel['max_tier'].fillna(0).astype(int)
panel['n_disasters'] = panel['n_disasters'].fillna(0).astype(int)

print(f'Panel shape : {panel.shape}')
print(f'States      : {panel["stateAbbreviation"].nunique()}')
print(f'Years       : {panel["incident_year"].min()} – {panel["incident_year"].max()}')
print('\nMax-tier distribution:')
TIER_NAMES = {
    0: 'No/Minor (<$1M)',
    1: 'Moderate ($1M–$50M)',
    2: 'Major ($50M–$500M)',
    3: 'Catastrophic (>$500M)'
}
for t, n in panel['max_tier'].value_counts().sort_index().items():
    print(f'  Tier {t} {TIER_NAMES[t]:<30} {n:>5,}  ({100*n/len(panel):.1f}%)')

## 5v2.2 Prospective Lag Features

All lag features use `shift(1)` — they only use data from **prior years**, never the current year.
This ensures the model is genuinely deployable at the start of a forecast year.

In [ ]:
panel = panel.sort_values(['stateAbbreviation', 'incident_year']).reset_index(drop=True)

# Prior 3-year disaster count
panel['prior_disasters_3yr'] = (
    panel.groupby('stateAbbreviation')['n_disasters']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).sum())
)

# Prior 3-year maximum tier
panel['prior_max_tier_3yr'] = (
    panel.groupby('stateAbbreviation')['max_tier']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).max())
)

# Prior year tier and count
panel['prior_year_tier'] = (
    panel.groupby('stateAbbreviation')['max_tier']
    .transform(lambda x: x.shift(1))
)
panel['prior_year_n_disasters'] = (
    panel.groupby('stateAbbreviation')['n_disasters']
    .transform(lambda x: x.shift(1))
)

lag_cols = ['prior_disasters_3yr', 'prior_max_tier_3yr',
            'prior_year_tier', 'prior_year_n_disasters']
for col in lag_cols:
    panel[col] = panel[col].fillna(0)

print('Lag features added.')
print(panel[['stateAbbreviation', 'incident_year', 'max_tier'] + lag_cols].head(8).to_string())

## 5v2.3 State-Level Demographics

Aggregate county-level demographics from the disaster data to state level.
Forward-fill within each state for years with no recorded disasters.

In [ ]:
demo_cols = ['population', 'median_income', 'poverty_rate', 'risk_score']

state_demos = (
    disas.groupby(['stateAbbreviation', 'incident_year'])[demo_cols]
    .median()
    .reset_index()
)

panel = panel.merge(state_demos, on=['stateAbbreviation', 'incident_year'], how='left')

# Forward-fill then back-fill within each state
panel = panel.sort_values(['stateAbbreviation', 'incident_year'])
panel[demo_cols] = (
    panel.groupby('stateAbbreviation')[demo_cols]
    .transform(lambda x: x.ffill().bfill())
)

# Fill any residual NAs with global median
for col in demo_cols:
    panel[col] = panel[col].fillna(panel[col].median())

print('Demographics merged and forward-filled.')
print(f'Null counts: {panel[demo_cols].isnull().sum().sum()} remaining')

## 5v2.4 Define Features & Target

**All features are prospective** — available at the start of the forecast year:
- `incident_year`: captures secular trend (inflation, policy changes)
- Lag features: disaster history of this state in prior years
- Demographics: state population, income, poverty, composite risk score

In [ ]:
CAT_FEATURES = ['stateAbbreviation']
NUM_FEATURES = [
    'incident_year',            # secular funding trend
    'prior_disasters_3yr',      # historical frequency (3yr window)
    'prior_max_tier_3yr',       # historical severity (3yr window)
    'prior_year_tier',          # most recent year outcome
    'prior_year_n_disasters',   # most recent year count
    'population',
    'median_income',
    'poverty_rate',
    'risk_score',
]
TARGET = 'max_tier'

CAT_FEATURES = [c for c in CAT_FEATURES if c in panel.columns]
NUM_FEATURES = [c for c in NUM_FEATURES if c in panel.columns]
FEATURES     = CAT_FEATURES + NUM_FEATURES

TARGET_NAMES = [TIER_NAMES[i] for i in range(4)]

df_model = panel[FEATURES + [TARGET]].dropna(subset=NUM_FEATURES).copy()
df_model[TARGET] = df_model[TARGET].astype(int)

print(f'Modeling rows: {len(df_model):,}  |  Features: {len(FEATURES)}')
print('Categorical:', CAT_FEATURES)
print('Numeric:    ', NUM_FEATURES)
print('\nClass distribution:')
for t, n in df_model[TARGET].value_counts().sort_index().items():
    print(f'  Tier {t} {TIER_NAMES[t]:<30} {n:>5,}  ({100*n/len(df_model):.1f}%)')

## 5v2.5 Train / Validation / Test Split

Same three-way temporal structure used throughout the project.
- **Train (pre-2016):** fit all models
- **Validation (2016–2017):** model selection — never use test set for selection
- **Test (2018+):** final reported metrics only

In [ ]:
VALIDATION_YEAR = 2016
SPLIT_YEAR      = 2018

train = df_model[df_model['incident_year'] <  VALIDATION_YEAR]
val   = df_model[(df_model['incident_year'] >= VALIDATION_YEAR) &
                 (df_model['incident_year'] <  SPLIT_YEAR)]
test  = df_model[df_model['incident_year'] >= SPLIT_YEAR]

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Train : {len(X_train):,}  ({int(train["incident_year"].min())}–{int(train["incident_year"].max())})')
print(f'Val   : {len(X_val):,}    ({int(val["incident_year"].min())}–{int(val["incident_year"].max())})')
print(f'Test  : {len(X_test):,}   ({int(test["incident_year"].min())}–{int(test["incident_year"].max())})')
print('\nTrain class distribution:')
for t, n in y_train.value_counts().sort_index().items():
    print(f'  Tier {t}: {n:>4,}  ({100*n/len(y_train):.1f}%)')

## 5v2.6 Preprocessing Pipeline

In [ ]:
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler())
])
preprocessor = ColumnTransformer([
    ('cat', cat_pipe, CAT_FEATURES),
    ('num', num_pipe, NUM_FEATURES)
])
print('Preprocessor defined.')

## 5v2.7 Train & Evaluate All Models

Fit on `X_train`, evaluate on `X_val` for selection and `X_test` for final reporting.
`class_weight='balanced'` addresses the heavy imbalance (most state-years have no major disaster).

In [ ]:
models = {
    'Baseline (Stratified)': DummyClassifier(strategy='stratified', random_state=42),
    'Logistic Regression':   LogisticRegression(max_iter=1000, class_weight='balanced',
                                                random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                    random_state=42, n_jobs=-1),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=200, random_state=42),
}

results_prospective = {}   # test metrics — final reporting
results_val         = {}   # val  metrics — model selection

for name, model in models.items():
    pipe = Pipeline([('pre', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)

    # Validation (model selection)
    val_preds = pipe.predict(X_val)
    m_val = classification_metrics(y_val.values, val_preds, label=name,
                                   target_names=TARGET_NAMES)
    results_val[name] = {**m_val, 'pipeline': pipe, 'preds': val_preds}

    # Test (final reporting only)
    test_preds = pipe.predict(X_test)
    m_test = classification_metrics(y_test.values, test_preds, label=name,
                                    target_names=TARGET_NAMES)
    m_test['pipeline'] = pipe
    m_test['preds']    = test_preds
    results_prospective[name] = m_test

    print(f'{name:<30}  val F1={m_val["F1_weighted"]:.4f}  |  test F1={m_test["F1_weighted"]:.4f}')

## 5v2.8 Hyperparameter Tuning — Gradient Boosting

RandomizedSearchCV with TimeSeriesSplit on the training set.
Best config selected on **validation F1**, not test.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

param_dist = {
    'model__n_estimators':      [100, 200, 300, 400],
    'model__max_depth':         [3, 4, 5, 6],
    'model__learning_rate':     [0.05, 0.1, 0.15, 0.2],
    'model__min_samples_split': [2, 5, 10],
    'model__subsample':         [0.7, 0.8, 1.0],
}

gb_base = Pipeline([
    ('pre',   preprocessor),
    ('model', GradientBoostingClassifier(random_state=42))
])

search = RandomizedSearchCV(
    gb_base, param_distributions=param_dist,
    n_iter=30, cv=tscv, scoring='f1_weighted',
    n_jobs=-1, random_state=42, verbose=1
)
search.fit(X_train, y_train)
print('\nBest params:', search.best_params_)
print(f'Best CV F1 (train folds): {search.best_score_:.4f}')

# Evaluate on validation (selection)
val_tuned   = search.best_estimator_.predict(X_val)
m_val_tuned = classification_metrics(y_val.values, val_tuned,
                                      label='GradBoosting (Tuned)',
                                      target_names=TARGET_NAMES)
results_val['GradBoosting (Tuned)'] = {**m_val_tuned,
                                        'pipeline': search.best_estimator_,
                                        'preds':    val_tuned}

# Evaluate on test (reporting)
test_tuned   = search.best_estimator_.predict(X_test)
m_test_tuned = classification_metrics(y_test.values, test_tuned,
                                       label='GradBoosting (Tuned)',
                                       target_names=TARGET_NAMES)
m_test_tuned['pipeline'] = search.best_estimator_
m_test_tuned['preds']    = test_tuned
results_prospective['GradBoosting (Tuned)'] = m_test_tuned

print(f'\nTuned — val  F1 : {m_val_tuned["F1_weighted"]:.4f}')
print(f'Tuned — test F1 : {m_test_tuned["F1_weighted"]:.4f}')

## 5v2.9 Stability Check — Bootstrap Resampling

In [ ]:
N_BOOTSTRAP = 50
rng = np.random.default_rng(42)
bootstrap_f1 = []

best_name_boot = max(results_val, key=lambda k: results_val[k]['F1_weighted'])
best_pipe      = results_val[best_name_boot]['pipeline']
print(f'Bootstrapping: {best_name_boot}')

for _ in range(N_BOOTSTRAP):
    idx = rng.choice(len(X_train), size=len(X_train), replace=True)
    pipe_boot = clone(best_pipe)
    pipe_boot.fit(X_train.iloc[idx], y_train.iloc[idx])
    preds_boot = pipe_boot.predict(X_test)
    bootstrap_f1.append(f1_score(y_test, preds_boot, average='weighted', zero_division=0))

bootstrap_f1 = np.array(bootstrap_f1)
print(f'\nBootstrap F1_weighted over {N_BOOTSTRAP} resamples:')
print(f'  Mean  : {bootstrap_f1.mean():.4f}')
print(f'  Std   : {bootstrap_f1.std():.4f}')
print(f'  95% CI: [{np.percentile(bootstrap_f1, 2.5):.4f},  {np.percentile(bootstrap_f1, 97.5):.4f}]')

plt.figure(figsize=(8, 3))
plt.hist(bootstrap_f1, bins=20, color='teal', edgecolor='white')
plt.axvline(bootstrap_f1.mean(), color='red', linestyle='--',
            label=f'Mean = {bootstrap_f1.mean():.3f}')
plt.axvline(np.percentile(bootstrap_f1, 2.5),  color='orange', linestyle=':', label='95% CI')
plt.axvline(np.percentile(bootstrap_f1, 97.5), color='orange', linestyle=':')
plt.title(f'Bootstrap F1_weighted — {best_name_boot} (Prospective)')
plt.xlabel('F1_weighted on held-out test set')
plt.legend()
plt.tight_layout()
plt.savefig('../data/processed/bootstrap_f1_prospective.png', dpi=150)
plt.show()

## 5v2.10 Results Summary — Validation vs Test

In [ ]:
summary = pd.DataFrame([
    {
        'Model':    name,
        'Val F1':   round(results_val[name]['F1_weighted'], 4) if name in results_val else '-',
        'Test F1':  round(results_prospective[name]['F1_weighted'], 4),
        'Test Acc': round(results_prospective[name]['Accuracy'], 4),
    }
    for name in results_prospective
]).set_index('Model')
display(summary)

# Per-class breakdown for best model
best_name = max(results_val, key=lambda k: results_val[k]['F1_weighted'])
print(f'\nPer-class metrics (test set) — {best_name}:')
report_df = pd.DataFrame(
    classification_report(
        y_test,
        results_prospective[best_name]['preds'],
        target_names=TARGET_NAMES,
        zero_division=0,
        output_dict=True
    )
).T.round(3)
display(report_df)

## 5v2.11 Feature Importances & Confusion Matrix

In [ ]:
rf_pipe  = results_prospective['Random Forest']['pipeline']
rf_model = rf_pipe.named_steps['model']
rf_pre   = rf_pipe.named_steps['pre']

ohe_names = rf_pre.transformers_[0][1].named_steps['ohe'].get_feature_names_out(CAT_FEATURES)
all_names = list(ohe_names) + NUM_FEATURES

importances = pd.Series(rf_model.feature_importances_, index=all_names)
importances.nlargest(15).sort_values().plot(
    kind='barh', figsize=(10, 5),
    title='Top 15 Feature Importances — Prospective Random Forest',
    color='teal'
)
plt.tight_layout()
plt.savefig('../data/processed/feature_importance_prospective.png', dpi=150)
plt.show()

# Confusion matrix for best model
best_name = max(results_val, key=lambda k: results_val[k]['F1_weighted'])
cm = confusion_matrix(y_test, results_prospective[best_name]['preds'])
fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=TARGET_NAMES)
disp.plot(ax=ax, colorbar=False, cmap='BuGn')
ax.set_title(f'Confusion Matrix — {best_name} (Prospective)')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix_prospective.png', dpi=150)
plt.show()

## 5v2.12 Prospective Risk Map — Test Year Sample

Demonstrate deployment: for each state, show predicted tier probabilities for the most recent test year.
This is exactly how a FEMA regional planner would use the model at the start of a fiscal year.

In [ ]:
best_name = max(results_val, key=lambda k: results_val[k]['F1_weighted'])
best_pipe = results_prospective[best_name]['pipeline']

# Use most recent test year as the demonstration forecast year
forecast_year = int(test['incident_year'].max())
X_forecast = test[test['incident_year'] == forecast_year][FEATURES]
states_forecast = test[test['incident_year'] == forecast_year]['stateAbbreviation'].values

if hasattr(best_pipe, 'predict_proba'):
    probs = best_pipe.predict_proba(X_forecast)
    risk_df = pd.DataFrame(probs, columns=[f'P(Tier {i})' for i in range(4)],
                           index=states_forecast)
    risk_df['Predicted Tier'] = best_pipe.predict(X_forecast)
    risk_df['Predicted Label'] = risk_df['Predicted Tier'].map(TIER_NAMES)
    risk_df = risk_df.sort_values('P(Tier 3)', ascending=False)
    print(f'Prospective risk forecast — Year {forecast_year}')
    print(f'Model: {best_name}')
    display(risk_df[['Predicted Label', 'P(Tier 2)', 'P(Tier 3)']].head(15).round(3))
else:
    preds = best_pipe.predict(X_forecast)
    risk_df = pd.DataFrame({'State': states_forecast,
                            'Predicted Tier': preds,
                            'Predicted Label': [TIER_NAMES[p] for p in preds]})
    display(risk_df.sort_values('Predicted Tier', ascending=False).head(15))

## 5v2.13 Save Best Model

In [ ]:
best_name = max(results_val, key=lambda k: results_val[k]['F1_weighted'])
print(f'Best model (val F1):  {best_name}')
print(f'  Val  F1_weighted : {results_val[best_name]["F1_weighted"]:.4f}')
print(f'  Test F1_weighted : {results_prospective[best_name]["F1_weighted"]:.4f}')

with open(PROCESSED + 'best_prospective_model.pkl', 'wb') as f:
    pickle.dump({
        'pipeline':     results_prospective[best_name]['pipeline'],
        'X_test':       X_test,
        'y_test':       y_test,
        'preds':        results_prospective[best_name]['preds'],
        'features':     FEATURES,
        'cat_features': CAT_FEATURES,
        'num_features': NUM_FEATURES,
        'target_names': TARGET_NAMES,
        'model_name':   best_name,
        'level':        'prospective',
        'tier_names':   TIER_NAMES,
    }, f)
print('Saved best_prospective_model.pkl')